In [1]:
from scipy.sparse import load_npz
from pathlib import Path

import pandas as pd
import numpy as np
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity

import joblib
import plotly.express as px
import matplotlib.pyplot as plt

In [2]:
processed_dir = Path("../data/processed")

matrice_tfidf = load_npz(processed_dir / "tfidf_matrix.npz")
ids = pd.read_csv(processed_dir / "tfidf_row_ids.csv")
data = pd.read_csv(processed_dir / "sample_clean.csv")

# Il faut s'assurer que la matrice TF-IDF et les identifiants des lignes correspondent
# bien aux produits du DataFrame original.
assert matrice_tfidf.shape[0] == len(ids)
assert ids["id"].is_unique
assert ids["id"].equals(data["id"])

print("Dimensions :", matrice_tfidf.shape)

Dimensions : (500, 1865)


In [3]:
dbscan = DBSCAN(
    eps=0.5,
    min_samples=5,
    metric="cosine"
)

labels = dbscan.fit_predict(matrice_tfidf)

# Les résultats restent associés aux identifiants exportés.
clusters = ids.copy()
clusters["cluster"] = labels

display(clusters["cluster"].value_counts().sort_index())

nb_clusters = clusters.loc[
    clusters["cluster"] != -1, "cluster"
].nunique()

print("Nombre de clusters :", nb_clusters)

cluster
-1    428
 0      5
 1      8
 2     22
 3      5
 4     15
 5      6
 6      6
 7      5
Name: count, dtype: int64

Nombre de clusters : 8


In [4]:
sans_bruit = labels != -1

X_eval = matrice_tfidf[sans_bruit]
labels_evalues = labels[sans_bruit]

nb_clusters = len(np.unique(labels_evalues))

score = silhouette_score(
    X_eval,
    labels_evalues,
    metric="cosine"
)

print(f"Silhouette hors bruit : {score:.3f}")
print(f"Produits évalués : {sans_bruit.sum()} / {len(labels)}")
print("Bruit (%) :", (labels == -1).mean() * 100)

Silhouette hors bruit : 0.456
Produits évalués : 72 / 500
Bruit (%) : 85.6


In [5]:
projection_2d = TruncatedSVD(n_components=2, random_state=42)

In [6]:
projection_2d

,"random_state random_state: int, RandomState instance or None, default=NoneUsed during randomized svd. Pass an int for reproducible results acrossmultiple function calls.See :term:`Glossary <random_state>`.",42
,"n_components n_components: int, default=2Desired dimensionality of output data.If algorithm='arpack', must be strictly less than the number of features.If algorithm='randomized', must be less than or equal to the number of features.The default value is useful for visualisation. For LSA, a value of100 is recommended.",2
,"algorithm algorithm: {'arpack', 'randomized'}, default='randomized'SVD solver to use. Either ""arpack"" for the ARPACK wrapper in SciPy(scipy.sparse.linalg.svds), or ""randomized"" for the randomizedalgorithm due to Halko (2009).",'randomized'
,"n_iter n_iter: int, default=5Number of iterations for randomized SVD solver. Not used by ARPACK. Thedefault is larger than the default in:func:`~sklearn.utils.extmath.randomized_svd` to handle sparsematrices that may have large slowly decaying spectrum.",5
,"n_oversamples n_oversamples: int, default=10Number of oversamples for randomized SVD solver. Not used by ARPACK.See :func:`~sklearn.utils.extmath.randomized_svd` for a completedescription... versionadded:: 1.1",10
,"power_iteration_normalizer power_iteration_normalizer: {'auto', 'QR', 'LU', 'none'}, default='auto'Power iteration normalizer for randomized SVD solver.Not used by ARPACK. See :func:`~sklearn.utils.extmath.randomized_svd`for more details... versionadded:: 1.1",'auto'
,"tol tol: float, default=0.0Tolerance for ARPACK. 0 means machine precision. Ignored by randomizedSVD solver.",0.0


In [7]:
coordonnees = projection_2d.fit_transform(matrice_tfidf)
visualisation = pd.DataFrame({
    "SVD 1": coordonnees[:, 0],
    "SVD 2": coordonnees[:, 1],
    "id": ids["id"].to_numpy(),
    "cluster": [
        "Bruit" if label == -1 else f"Cluster {label}"
        for label in labels
    ]
})

In [8]:
coeurs = np.zeros(len(labels), dtype=bool)
coeurs[dbscan.core_sample_indices_] = True

visualisation["statut"] = np.where(
    labels == -1,
    "Bruit",
    np.where(coeurs, "Cœur", "Bordure")
)

fig = px.scatter(
    visualisation,
    x="SVD 1",
    y="SVD 2",
    color="cluster",
    symbol="statut",
    symbol_map={"Cœur": "circle", "Bordure": "diamond", "Bruit": "x"},
    color_discrete_map={"Bruit": "lightgray"},
    hover_data=["id", "statut"],
    opacity=0.7,
    title="DBSCAN — clusters et vrais points cœur",
    height=600,
    width=600
)

fig.show()

In [9]:
tsne = TSNE(
    n_components=2,
    metric="cosine",
    perplexity=30,
    init="random",
    learning_rate="auto",
    random_state=42
)

coord_tsne = tsne.fit_transform(matrice_tfidf.toarray())

visu_tsne = visualisation.copy()
visu_tsne["t-SNE 1"] = coord_tsne[:, 0]
visu_tsne["t-SNE 2"] = coord_tsne[:, 1]

fig = px.scatter(
    visu_tsne,
    x="t-SNE 1",
    y="t-SNE 2",
    color="cluster",
    symbol="statut",
    symbol_map={"Cœur": "circle", "Bordure": "diamond", "Bruit": "x"},
    color_discrete_map={"Bruit": "lightgray"},
    hover_data=["id", "statut"],
    opacity=0.7,
    title="DBSCAN — visualisation t-SNE, distance cosinus",
    height=600,
    width=600
)

fig.show()

In [10]:
tsne_3d = TSNE(
    n_components=3,
    metric="cosine",
    perplexity=30,
    init="random",
    learning_rate="auto",
    random_state=42
)

coord_3d = tsne_3d.fit_transform(matrice_tfidf.toarray())

visu_3d = visualisation.copy()
visu_3d[["t-SNE 1", "t-SNE 2", "t-SNE 3"]] = coord_3d

fig = px.scatter_3d(
    visu_3d,
    x="t-SNE 1",
    y="t-SNE 2",
    z="t-SNE 3",
    color="cluster",
    symbol="statut",
    symbol_map={"Cœur": "circle", "Bordure": "diamond", "Bruit": "x"},
    color_discrete_map={"Bruit": "lightgray"},
    hover_data=["id", "statut"],
    opacity=0.7,
    title="DBSCAN — projection t-SNE en 3D",
    height=600,
    width=600
)

fig.update_traces(marker_size=4)
fig.update_layout(scene_aspectmode="data")
fig.show()

In [11]:
vectorizer = joblib.load("../models/tfidf_vectorizer.joblib")
mots = vectorizer.get_feature_names_out()

# Retrouver les descriptions dans l'ordre exact de la matrice
source = pd.read_csv(
    processed_dir / "sample_clean.csv"
).set_index("id")

lecture = ids.copy()
lecture["cluster"] = labels
lecture["texte"] = source.loc[
    ids["id"], "description_texte"
].to_numpy()

assert matrice_tfidf.shape == (len(lecture), len(mots))

In [12]:
cluster_ids = np.unique(labels[labels != -1]) # Parcours tous les clusters sauf le bruit (-1)

for cluster_id in cluster_ids:
    masque = labels == cluster_id
    assert cluster_id != -1 and masque.any()

    X_cluster = matrice_tfidf[masque]

    # Poids TF-IDF moyen de chaque mot dans ce groupe
    poids = np.asarray(X_cluster.mean(axis=0)).ravel()
    top = np.argsort(poids)[::-1][:12]
    top = top[poids[top] > 0]

    mots_cluster = pd.DataFrame({
        "mot": mots[top],
        "poids TF-IDF moyen": poids[top]
    })

    fig = px.bar(
        mots_cluster,
        x="poids TF-IDF moyen",
        y="mot",
        orientation="h",
        title=f"Cluster {cluster_id} — mots dominants",
        height=400,
        width=600
    )
    fig.update_yaxes(categoryorder="total ascending")
    fig.show()

    # Descriptions les plus proches du centre moyen en distance cosinus
    centre = np.asarray(X_cluster.mean(axis=0))
    similarites = cosine_similarity(X_cluster, centre).ravel()

    positions = np.flatnonzero(masque)
    selection = positions[np.argsort(similarites)[::-1][:3]]

    print(f"Cluster {cluster_id} : {len(positions)} produits")

    with pd.option_context("display.max_colwidth", None):
        display(lecture.iloc[selection][["id", "texte"]])

Cluster 0 : 5 produits


,id,texte
359,360,"cap 2 t - shirt - like sweet breeze long runs hot days , capilene 2 features open knit invites airflow cool provides excellent wicking exceptional breathability dry . insulative chilly days winter hasn t released grip , s fastest - drying performance baselayer fabrics . feminine neck elbow - length sleeves stay . raglan - style sleeves flat - ribbed underarm panels merge way straps eliminate chafe . 4 - oz 100 % polyester 54 % recycled gladiodor natural odor control garment . recyclable common threads recycling program ."
21,22,"cap 2 t - shirt - revels sweat jamborees hill climbs , ridge runs desert singletrack resolute sun . fastest - drying performance baselayer fabrics , open knit invites airflow provides excellent wicking exceptional breathability . worn beneath layers , s insulative chilly days . self - binding neck pliable soft raglan sleeves flat - ribbed underarm panels merge way straps eliminate chafe . 4 - oz 100 % polyester 54 % recycled gladiodor natural odor control garment . recyclable common threads recycling program ."
358,359,"cap 2 cap sleeve - fastest - drying performance baselayer fabrics , capilene 2 open knit invites airflow cool provides excellent wicking exceptional breathability dry . s insulative chilly days winter hasn t released grip . minimalist tech features feminine v - neck self - fabric neckline , easy - moving underarm panels provide shaping . raglan - style sleeves stay way pack straps eliminate chafe . 4 - oz 100 % polyester 54 % recycled gladiodor natural odor control garment . recyclable common threads recycling program ."


Cluster 1 : 8 produits


,id,texte
68,69,"lead examined life poster - guanacos standing sentry patagonia , chile . 30 years ve paid tribute human adventures near far collection photographs speak passion , joy beauty lives spent outside . photos appeared exclusively web site , catalog walls retail stores specialty outdoor clothiers world . excited select photos available posters . hope brighten space inspire outside ."
237,238,"wyoming climbing poster - max factor , 5.11c . vedauwoo , wyoming . 30 years ve paid tribute human adventures near far collection photographs speak passion , joy beauty lives spent outside . photos appeared exclusively web site , catalog walls retail stores specialty outdoor clothiers world . excited select photos available posters . hope brighten space inspire outside ."
489,490,"flyfishing athabasca poster - patrik ondrus wets line athabasca river . jasper national park , alberta , canada . 30 years ve paid tribute human adventures near far collection photographs speak passion , joy beauty lives spent outside . photos appeared exclusively web site , catalog walls retail stores specialty outdoor clothiers world . excited select photos available posters . hope brighten space inspire outside ."


Cluster 2 : 22 produits


,id,texte
57,58,73 logo t - shirt - soft - wearing ringspun organic cotton screen - print inks pvc- phthalate - free
63,64,gpiw classic t - shirt - soft - wearing ringspun organic cotton screen - print inks pvc- phthalate - free
431,432,girl s live simply deer t - shirt - soft - wearing ringspun organic cotton screen - print inks pvc- phthalate - free


... (6 outputs left out)

Cluster 6 : 6 produits


,id,texte
288,289,"owlin t - shirt - owlin t - shirt wise choice flights fancy limb . artist jason stowell s design printed 100 % ringspun organic cotton pvc- phthalate - free inks . lightweight shirt s anika fit features longer torso s form - fitting , shorter sleeves wide , deep neckline . taped shoulder seams extra soft . recyclable common threads recycling program ."
314,315,"kite town t - shirt - artist chris del moro transforms wind fun whimsical kite town t - shirt . printed lightweight , 100 % organic cotton pvc- phthalate - free inks , tee taped shoulder seams soft - - skin . shirt s anika fit features longer torso s form - fitting , shorter sleeves wide , deep neckline . recyclable common threads recycling program ."
166,167,"birdwalk t - shirt - temporarily grounded , artist chris del moro s scandinavian - inspired birds elegantly stretch legs birdwalk t - shirt . 100 % organic cotton , shirt s anika fit features longer torso s form - fitting , shorter sleeves wide , deep neckline . taped shoulder seams ringspun fabric yarns soft - - skin . lightweight tee recyclable common threads recycling program ."


Cluster 7 : 5 produits


,id,texte
440,441,"cap 3 crew - capilene 3 , versatile fastest - wicking performance baselayer fabrics , helps stay dry warm . crew provides excellent insulation dense interior honeycomb grid s brushed warmth , softness compressibility . smooth jersey exterior glides readily beneath layers . raglan sleeves single - piece shoulder panels merge way shoulder straps , panels offset eliminate chafe . 4.8 - oz 100 % polyester 64 % recycled gladiodor natural odor control garment . recyclable common threads recycling program ."
439,440,"cap 3 bottoms - unwavering foundation cold - weather clothing system , bottoms dry , warm moving forward . capilene 3 , versatile fastest - wicking performance baselayer fabrics providing excellent insulation ultimate comfort skin , dense interior honeycomb grid brushed warmth , softness compressibility . smooth jersey exterior glides readily beneath layers , brushed , scalloped - edge elastic waistband , gusseted crotch flat - seam construction eliminates chafe . 4.8 - oz 100 % polyester 64 % recycled gladiodor natural odor control garment . recyclable common threads recycling program ."
442,443,"cap 3 crew - fundamental funhog lifestyle , crew nonchalantly flows turns bowls catching ride depot . capilene 3 versatile fastest - wicking performance baselayer fabrics . provides excellent insulation ultimate comfort skin . dense interior honeycomb construction brushed warmth , softness compressibility , smooth jersey exterior glides readily beneath layers . details include rib collar , raglan sleeves single - piece shoulder panels merge way shoulder straps . 4.8 - oz 100 % polyester 64 % recycled gladiodor natural odor control garment . recyclable common threads recycling program ."
